# FORMS Sample Mission — IGRF-13 B-field along an SSO

Propagates one orbit of a 500 km Sun-synchronous spacecraft and samples
the geomagnetic field at every 60 s step using the `BIGRFSampler` routine.

## 1. Imports

In [1]:
import sys
import math

# Add project root so `routine` is importable when the kernel
# is launched from the notebook/ sub-folder.
import pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import forms
from routine import BIGRFSampler

## 2. Mission definition

500 km circular SSO (i ≈ 97.4°).  
One orbit ≈ 5 670 s → we run for 5 760 s (96 steps at 60 s).

In [2]:
# Orbit parameters
R_EARTH_KM = 6371.0
ALT_KM     = 500.0
SMA_KM     = R_EARTH_KM + ALT_KM   # 6871 km
INC_DEG    = 97.4                   # SSO inclination
EPOCH      = "2026:01:01:00:00:00"
DT_S       = 60.0                   # propagation step (s)
DURATION_S = 5760.0                 # ~1 orbit

N_STEPS = int(DURATION_S / DT_S)
print(f"SMA = {SMA_KM} km   inc = {INC_DEG}°   steps = {N_STEPS}")

SMA = 6871.0 km   inc = 97.4°   steps = 96


## 3. Initialise FORMS handle

In [3]:
f = forms.FORMS()

f.load_koe(
    SMA_KM, 0.001, INC_DEG,
    raan=0.0, aop=0.0, ta=0.0,
    timestamp=EPOCH,
)

f.time.set_span("simulated", duration=DURATION_S, units="seconds")

(
    f.satellite.propagator_config
    .set_type("rk45")
    .set_mode("simulated", fixed_dt=DT_S)
    .apply()
)

print("Epoch  :", f.time.timestamp)
print("epoch0 :", f.time.epoch0, "MJD2000")
print("epoch1 :", f.time.epoch1, "MJD2000")

[2026-06-30 04:00:06.377] [FORMS] [WARNING] EOP data not available - transforms will use default values
[2026-06-30 04:00:06.383] [Propagator] [INFO] Force model: <ForceModel gravity=point_mass>
Epoch  : 2026:01:01:00:00:0.0000
epoch0 : 9496.5 MJD2000
epoch1 : 9496.566666666668 MJD2000


## 4. BIGRF sampling routine

`BIGRFSampler` records `BGCRS` (inertial) and `BLocal` (geocentric spherical)
at every call to `.record()`, plus the sub-satellite lat / lon / alt.

In [4]:
sampler = BIGRFSampler(f)

# Derive initial state before the first step
f.derive()
sampler.record()

for _ in range(N_STEPS):
    f.satellite.step()
    f.derive()
    sampler.record()

print(f"Collected {len(sampler)} samples")

Collected 97 samples


## 5. Inspect results

In [5]:
records = sampler.to_records()

# Print header + first few rows
keys = list(records[0].keys())
print("  ".join(f"{k:>18s}" for k in keys))
print("-" * (20 * len(keys)))
for row in records[:6]:
    print("  ".join(f"{row[k]:>18.3f}" for k in keys))

               t_s             lat_deg             lon_deg              alt_km          Bx_gcrs_nT          By_gcrs_nT          Bz_gcrs_nT         Br_local_nT     Btheta_local_nT       Bphi_local_nT             Bmag_nT
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
             0.000               0.146            -100.328             485.992           -6977.156            2189.253           22684.210           -6977.156          -22684.279            2188.536           23833.734
            60.000              -3.664            -101.071             486.094           -2281.784            2505.664           22751.856           -3800.734          -22550.217            2484.750           23002.866
           120.000              -7.473            -101.818             486.412            2302.736            2758.273    

## 6. Plot |B| over one orbit

In [6]:
import matplotlib.pyplot as plt

t    = [r["t_s"]     for r in records]
bmag = [r["Bmag_nT"] for r in records]
lat  = [r["lat_deg"] for r in records]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(t, bmag, color="steelblue")
ax1.set_ylabel("|B| (nT)")
ax1.set_title("IGRF-13 field strength — 500 km SSO, one orbit")
ax1.grid(True, alpha=0.3)

ax2.plot(t, lat, color="darkorange")
ax2.set_ylabel("Latitude (°)")
ax2.set_xlabel("Time since epoch (s)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 7. Plot GCRS components

In [ ]:
bx = [r["Bx_gcrs_nT"] for r in records]
by = [r["By_gcrs_nT"] for r in records]
bz = [r["Bz_gcrs_nT"] for r in records]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t, bx, label="Bx")
ax.plot(t, by, label="By")
ax.plot(t, bz, label="Bz")
ax.set_ylabel("B GCRS (nT)")
ax.set_xlabel("Time since epoch (s)")
ax.set_title("GCRS magnetic-field components")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()